In [12]:
import pyrealsense2 as rs
import json
import datetime


OUTPUT_FILE = "realsense_camera_dump.json"


def distortion_to_dict(intr):
    return {
        "model": str(intr.model),
        "coefficients": list(intr.coeffs)
    }


def intrinsics_to_dict(intr):

    return {
        "width": intr.width,
        "height": intr.height,

        "fx": intr.fx,
        "fy": intr.fy,

        "cx": intr.ppx,
        "cy": intr.ppy,

        "distortion": distortion_to_dict(intr)
    }


def extrinsics_to_dict(ext):

    return {

        "rotation_matrix": list(ext.rotation),

        "translation_meters": list(ext.translation)

    }



def get_camera_info(device):

    info = {}

    for item in [
        rs.camera_info.name,
        rs.camera_info.serial_number,
        rs.camera_info.firmware_version,
        rs.camera_info.product_id,
        rs.camera_info.product_line,
        rs.camera_info.usb_type_descriptor
    ]:

        try:
            info[str(item)] = device.get_info(item)

        except:
            pass

    return info



def get_sensor_options(sensor):

    options = {}

    try:

        for option in sensor.get_supported_options():

            try:

                options[str(option)] = {

                    "value":
                        sensor.get_option(option),

                    "range":
                    {
                        "min":
                            sensor.get_option_range(option).min,

                        "max":
                            sensor.get_option_range(option).max,

                        "step":
                            sensor.get_option_range(option).step,

                        "default":
                            sensor.get_option_range(option).def_

                    }

                }

            except Exception:

                pass


    except Exception:
        pass


    return options




def main():


    ctx = rs.context()

    devices = ctx.query_devices()


    if len(devices) == 0:

        print("Nie znaleziono kamery RealSense")

        return


    device = devices[0]


    data = {}


    data["timestamp"] = str(datetime.datetime.now())


    # INFORMACJE O URZĄDZENIU

    data["device"] = get_camera_info(device)



    # SENSORY

    data["sensors"] = {}



    for sensor in device.query_sensors():

        name = sensor.get_info(rs.camera_info.name)


        sensor_data = {}


        sensor_data["options"] = get_sensor_options(sensor)


        data["sensors"][name] = sensor_data



    # PROFILE STREAMÓW

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 1920, 1080, rs.format.rgb8, 30)

    profile = pipeline.start(config)



    data["streams"] = {}



    for stream in profile.get_streams():

        try:

            video = stream.as_video_stream_profile()


            stream_name = str(stream.stream_type())


            intr = video.get_intrinsics()


            data["streams"][stream_name] = {


                "intrinsics":

                    intrinsics_to_dict(intr),


                "fps":

                    video.fps(),


                "format":

                    str(video.format()),


            }


        except Exception:

            pass




    # EXTRINSICS
    data["extrinsics"] = {}



    profiles = profile.get_streams()


    for src in profiles:


        for dst in profiles:


            if src != dst:


                try:

                    ext = src.get_extrinsics_to(dst)


                    key = (
                        str(src.stream_type())
                        +
                        "_TO_"
                        +
                        str(dst.stream_type())
                    )


                    data["extrinsics"][key] = \
                        extrinsics_to_dict(ext)


                except:

                    pass



    pipeline.stop()




    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:


        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )



    print(
        "Zapisano:",
        OUTPUT_FILE
    )



if __name__ == "__main__":

    main()

Zapisano: realsense_camera_dump.json
